# 6. Transfer Learning 실험

이 노트북은 `05_02_ResNet_계열_확장.ipynb` 다음 단계로, **이미 학습된 CNN을 가져와 새로운 문제에 재사용하는 방법** 을 실습합니다.

앞선 노트북에서는 ResNet 계열이 어떻게 깊이(depth), 너비(width), 경로 수(cardinality)를 늘려 가며 확장되는지 살펴봤습니다. 이번에는 그 구조를 직접 처음부터 학습시키는 대신, **대규모 데이터셋에서 미리 학습된 모델(pretrained model)** 을 가져와 CIFAR-10 분류에 적용해 보겠습니다.

이번 노트북의 목표는 다음과 같습니다.

- transfer learning이 왜 자주 쓰이는지 이해합니다.
- `feature extractor` 방식과 `fine-tuning` 방식의 차이를 비교합니다.
- `pretrained ResNet-18`의 마지막 분류기(`fc`)를 CIFAR-10에 맞게 바꿔 봅니다.
- 같은 데이터에서도 학습 전략에 따라 어떤 차이가 생기는지 확인합니다.


## 6-1. 왜 Transfer Learning을 사용할까?

실제로는 새로운 데이터셋이 항상 충분히 크지 않습니다. 처음부터 큰 CNN을 전부 학습하려면 시간이 오래 걸리고, 데이터가 부족하면 쉽게 과적합될 수도 있습니다.

이럴 때 많이 쓰는 방법이 transfer learning입니다. 핵심 아이디어는 다음과 같습니다.

- 큰 데이터셋(ImageNet 등)에서 이미 학습된 모델은 기본적인 시각 특징을 잘 추출합니다.
- 새로운 문제에서는 그 특징 추출 능력을 재사용하고, 마지막 분류기만 바꾸거나 일부 층만 다시 학습합니다.
- 그래서 적은 데이터, 짧은 시간으로도 꽤 좋은 출발점을 얻을 수 있습니다.

즉, transfer learning은 **처음부터 전부 새로 배우는 것** 이 아니라, **이미 배운 시각 지식을 다른 문제로 옮겨 오는 것** 이라고 볼 수 있습니다.


## 6-2. 두 가지 대표 전략

보통 transfer learning은 두 방식으로 많이 나뉩니다.

- **Feature Extractor**: backbone의 가중치는 고정(freeze)하고, 마지막 분류기만 학습합니다.
- **Fine-Tuning**: backbone 일부 또는 전체를 함께 업데이트합니다.

두 방식의 차이는 다음처럼 이해하면 좋습니다.

- feature extractor는 빠르고 안정적이지만, 새 데이터셋에 맞춘 적응 폭은 상대적으로 작습니다.
- fine-tuning은 더 잘 맞춰질 가능성이 있지만, 학습 시간이 길고 learning rate 조절도 더 중요합니다.

이번 노트북에서는 두 방식을 같은 데이터셋으로 직접 비교해 봅니다.


In [ ]:
# 필요 라이브러리가 없다면 아래 주석을 해제해서 설치하세요.
# !pip install torch torchvision matplotlib


In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)


## 6-3. 데이터 준비

pretrained ResNet은 원래 ImageNet 기준 입력 크기인 `224 x 224`에서 많이 사용되므로, 이번에도 CIFAR-10 이미지를 확대해서 넣습니다. 실습 시간을 줄이기 위해 기본값은 subset으로 설정해 두었습니다.


In [ ]:
image_size = 224
train_size = 4000
val_size = 1000
batch_size = 32

mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

full_train_aug = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
full_train_eval = datasets.CIFAR10(root='./data', train=True, download=False, transform=eval_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=eval_transform)

classes = full_train_aug.classes

generator = torch.Generator().manual_seed(42)
all_indices = torch.randperm(len(full_train_aug), generator=generator).tolist()
train_indices = all_indices[:train_size]
val_indices = all_indices[train_size:train_size + val_size]

train_dataset = Subset(full_train_aug, train_indices)
val_dataset = Subset(full_train_eval, val_indices)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size * 2, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size * 2, shuffle=False)

print('Classes:', classes)
print('Train samples:', len(train_dataset))
print('Validation samples:', len(val_dataset))
print('Test samples:', len(test_dataset))


In [ ]:
def denormalize(image):
    mean_tensor = torch.tensor(mean).view(3, 1, 1)
    std_tensor = torch.tensor(std).view(3, 1, 1)
    return (image.cpu() * std_tensor + mean_tensor).clamp(0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(denormalize(image).permute(1, 2, 0))
    ax.set_title(classes[label])
    ax.axis('off')
plt.tight_layout()
plt.show()


## 6-4. Pretrained ResNet-18 불러오기

이번 실습에서는 `ResNet-18`을 사용합니다. 구조가 너무 무겁지 않으면서도 transfer learning 흐름을 이해하기에 적당하기 때문입니다.

중요한 점은 마지막 `fc` 층이 원래 ImageNet의 1000개 클래스를 출력하도록 되어 있다는 점입니다. 따라서 CIFAR-10에 맞게 `10개 클래스 출력`으로 바꿔야 합니다.

아래 `use_pretrained=True`이면 ImageNet 가중치를 사용합니다. 처음 실행할 때는 가중치 다운로드가 필요할 수 있습니다.


In [ ]:
use_pretrained = True

weights = models.ResNet18_Weights.DEFAULT if use_pretrained else None
model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, len(classes))

print(model)


In [ ]:
total_params = sum(param.numel() for param in model.parameters())
trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)

print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print('마지막 fc 입력 차원:', model.fc.in_features)


## 6-5. Feature Extractor 모델 만들기

먼저 backbone을 고정하고 마지막 `fc`만 학습하는 버전을 만듭니다. 이 방식은 보통 빠르고 안정적이며, 데이터가 적을 때 좋은 출발점이 됩니다.


In [ ]:
def build_feature_extractor_model(num_classes=10, use_pretrained=True):
    weights = models.ResNet18_Weights.DEFAULT if use_pretrained else None
    model = models.resnet18(weights=weights)

    for param in model.parameters():
        param.requires_grad = False

    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


feature_model = build_feature_extractor_model(num_classes=len(classes), use_pretrained=use_pretrained)
trainable_params = sum(param.numel() for param in feature_model.parameters() if param.requires_grad)
print('Feature Extractor 학습 파라미터 수:', f'{trainable_params:,}')


## 6-6. Fine-Tuning 모델 만들기

이번에는 마지막 분류기뿐 아니라 backbone도 함께 업데이트하는 버전을 만듭니다. 다만 처음부터 전체를 크게 흔들면 pretrained 가중치의 장점을 잃을 수 있으므로, 보통은 작은 learning rate를 사용하는 편이 좋습니다.

여기서는 두 가지 fine-tuning 방식 중 가장 단순한 **전체 모델 fine-tuning** 을 사용합니다.


In [ ]:
def build_finetune_model(num_classes=10, use_pretrained=True):
    weights = models.ResNet18_Weights.DEFAULT if use_pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


finetune_model = build_finetune_model(num_classes=len(classes), use_pretrained=use_pretrained)
trainable_params = sum(param.numel() for param in finetune_model.parameters() if param.requires_grad)
print('Fine-Tuning 학습 파라미터 수:', f'{trainable_params:,}')


파라미터 수를 비교해 보면, feature extractor는 대부분의 backbone을 고정하기 때문에 실제로 업데이트되는 파라미터가 훨씬 적습니다. 반면 fine-tuning은 더 많은 자유도를 가지는 대신, 학습 시간과 튜닝 부담도 커집니다.


## 6-7. 학습 함수 정의

두 전략을 같은 방식으로 비교하기 위해 공통 학습 함수를 정의합니다. CPU 환경이라면 먼저 `epochs = 1` 정도로 시작하는 것이 좋습니다.


In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, epochs=1, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    history = []

    model = model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history.append((train_loss, train_acc, val_loss, val_acc))

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

    model.load_state_dict(best_state)
    return model, history, criterion


## 6-8. Feature Extractor vs Fine-Tuning 실험

기본값으로는 두 모델 모두 실행되도록 두었습니다. 학습 시간이 부담되면 `run_finetune = False`로 두고 feature extractor만 먼저 실험해도 됩니다.


In [ ]:
run_feature_extractor = True
run_finetune = True

feature_epochs = 1
finetune_epochs = 1

feature_lr = 0.001
finetune_lr = 0.0001

results = {}
trained_models = {}

if run_feature_extractor:
    print('=== Feature Extractor 학습 ===')
    feature_model = build_feature_extractor_model(num_classes=len(classes), use_pretrained=use_pretrained)
    feature_model, feature_history, feature_criterion = train_model(
        feature_model,
        train_loader,
        val_loader,
        epochs=feature_epochs,
        lr=feature_lr
    )
    feature_test_loss, feature_test_acc = evaluate(feature_model, test_loader, feature_criterion)
    trained_models['feature_extractor'] = feature_model
    results['feature_extractor'] = {
        'history': feature_history,
        'test_loss': feature_test_loss,
        'test_acc': feature_test_acc,
    }
    print(f'Feature Extractor | Test Loss: {feature_test_loss:.4f} | Test Acc: {feature_test_acc:.4f}')
    print()

if run_finetune:
    print('=== Fine-Tuning 학습 ===')
    finetune_model = build_finetune_model(num_classes=len(classes), use_pretrained=use_pretrained)
    finetune_model, finetune_history, finetune_criterion = train_model(
        finetune_model,
        train_loader,
        val_loader,
        epochs=finetune_epochs,
        lr=finetune_lr
    )
    finetune_test_loss, finetune_test_acc = evaluate(finetune_model, test_loader, finetune_criterion)
    trained_models['fine_tuning'] = finetune_model
    results['fine_tuning'] = {
        'history': finetune_history,
        'test_loss': finetune_test_loss,
        'test_acc': finetune_test_acc,
    }
    print(f'Fine-Tuning      | Test Loss: {finetune_test_loss:.4f} | Test Acc: {finetune_test_acc:.4f}')


In [ ]:
if results:
    model_names = list(results.keys())
    test_accs = [results[name]['test_acc'] for name in model_names]

    plt.figure(figsize=(7, 4))
    plt.bar(model_names, test_accs, color=['#2563eb', '#0f766e'])
    plt.ylim(0, 1)
    plt.ylabel('Accuracy')
    plt.title('Transfer Learning 전략별 Test Accuracy 비교')
    plt.show()

    plt.figure(figsize=(8, 4))
    for model_name in model_names:
        history = results[model_name]['history']
        epochs_axis = range(1, len(history) + 1)
        train_accs = [item[1] for item in history]
        val_accs = [item[3] for item in history]
        plt.plot(epochs_axis, train_accs, marker='o', label=f'{model_name} Train')
        plt.plot(epochs_axis, val_accs, marker='s', linestyle='--', label=f'{model_name} Val')

    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Epoch별 Accuracy 추이')
    plt.legend()
    plt.show()
else:
    print('실행된 실험이 없어서 결과 그래프를 그리지 않습니다.')


짧은 실험에서는 feature extractor가 빠르게 안정적인 성능을 내는 경우가 많고, fine-tuning은 epoch를 조금 더 주면 점점 더 좋아지는 경우가 많습니다. 물론 이는 데이터 크기와 learning rate 설정에 따라 달라질 수 있습니다.


## 6-9. 예측 결과 확인

마지막으로 더 좋은 성능을 낸 모델이 실제 테스트 이미지에서 어떤 예측을 하는지 확인합니다.


In [ ]:
if results:
    best_model_name = max(results, key=lambda name: results[name]['test_acc'])
    best_model = trained_models[best_model_name]
    best_model.eval()

    images, labels = next(iter(test_loader))
    images = images.to(device)
    labels = labels.to(device)

    with torch.no_grad():
        outputs = best_model(images)
        preds = outputs.argmax(dim=1)

    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    for ax, image, label, pred in zip(axes.flat, images[:8], labels[:8], preds[:8]):
        ax.imshow(denormalize(image).permute(1, 2, 0))
        ax.set_title(f'T: {classes[label]}\nP: {classes[pred]}')
        ax.axis('off')
    plt.suptitle(f'Best Model: {best_model_name}', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('학습을 실행하지 않아 예측 결과를 표시하지 않습니다.')


## 6-10. 추가 실험 아이디어

이번 실습을 바탕으로 다음 실험도 이어서 해 볼 수 있습니다.

- `use_pretrained=False`로 바꿔서 pretrained 가중치의 효과를 직접 비교하기
- `layer4`만 열어 두고 나머지는 freeze하는 부분 fine-tuning 실험
- `ResNet-18` 대신 `ResNet-50` 또는 `EfficientNet`으로 backbone 교체
- 데이터 증강을 더 강화해 fine-tuning 성능이 얼마나 달라지는지 보기


## 정리

이번 노트북의 핵심은 다음과 같습니다.

- transfer learning은 이미 학습된 시각 특징을 새로운 문제에 재사용하는 방법입니다.
- feature extractor는 빠르고 간단하며, 작은 데이터셋에서 좋은 출발점이 됩니다.
- fine-tuning은 더 많은 적응 능력을 가지지만, 학습률과 epoch 설정이 더 중요합니다.
- pretrained CNN을 활용하면 처음부터 큰 모델을 새로 학습하는 부담을 크게 줄일 수 있습니다.

다음 단계로는 `Grad-CAM` 같은 시각화 기법을 통해, pretrained ResNet이 이미지의 어느 부분을 보고 판단하는지 확인해 보면 좋습니다.
